# Tools

# 1. Create tools

## 1.1. Basic tool definition

- Required as they define the tool’s input schema. 
- The docstring should be informative and concise to help the model understand the tool’s purpose.

Note: khi gọi tools thì tạo một tool schema rồi gửi cho LLM, docstring = description

```python
{
  "name": "get_weather",
  "description": "Get the current weather for a city.",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "Name of the city."
      },
      "unit": {
        "type": "string",
        "description": "Temperature unit, either celsius or fahrenheit.",
        "default": "celsius"
      }
    },
    "required": ["city"]
  }
}
```

In [1]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

C:\Users\Mario\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2. Customize tool properties

In [2]:
@tool("web_search")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)  # web_search

web_search


In [ ]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

print(calc.name + ": " + calc.description)  # calculator: Performs arithmetic calculations. Use this for any math problems.

calculator: Performs arithmetic calculations. Use this for any math problems.


AttributeError: 'StructuredTool' object has no attribute 'config'

In [15]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

# 2. Access context

Tools are most powerful when they can access runtime information like conversation history, user data, and persistent memory.

Tools can access runtime information through the ToolRuntime parameter



```python
ToolRuntime(
  self,
  state: StateT,
  context: ContextT,
  config: RunnableConfig,
  stream_writer: StreamWriter,
  tool_call_id: str | None,
  store: BaseStore | None,
  tools: list[BaseTool] = list(),
  execution_info: ExecutionInfo | None = None,
  server_info: ServerInfo | None = None
)

state: The current graph state
tool_call_id: The ID of the current tool call
config: RunnableConfig for the current execution
context: Runtime context (shared with Runtime)
store: BaseStore instance for persistent storage (shared with Runtime)
stream_writer: StreamWriter for streaming output (shared with Runtime)
tools: List of all available BaseTool instances

```

## 2.1. Short-term memory (State)

In [17]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")